# Step 1 - Prototype 2

We now have our first working prototype for step 1. In the second (and probably final) one, we mainly want to fix some smaller issues and decide on foundational formulations / implementations that will make extendability much easier later on. These are:
- Think of a good way to encode and implement train stations (probably already along with the corresponding constraints (essentially blocking movement until the leave time)).
- Adjust the optimization goal to minimize lateness (for now only at the destination station). Additionally, maybe make sure that the formulation can be extended in terms of travel and dwell time minimization goals later.

Adding a safety distance now might also be doable, however, this is in a sense then also connected to different train speeds (that might require different safety distances) and this is probably really then a point for step 2...

In [1]:
import gurobipy as gp
from gurobipy import GRB

import numpy as np
from enum import Enum

In [2]:
# Initialize the Model
model = gp.Model("Basic_MILP")

Restricted license - for non-production use only - expires 2026-11-23


### Define the track system and timetable

Here we want to see, if it makes sense to create (separate?) definitions for train stations and block segments that can then hopefully be combined automatically or, if just adding segments of length one with a given 'station capacity' should work just fine (or even better xD).

Given the fact, that we probably only want to have leave and arrival times at stations and not on 'random' points on the tracks, a more native way of distinguishing (separate blocks and train stations) might be beneficial. This could then be handled via an additional boolean array `is_train_station_array` (or something like that xD).

**Another Idea:** `segment_lengths` is a list of lists (and single entries?). The same goes for segment capacities and per construction, there is always a station (with capacities specified in `station_capacities`) between a given connecting segment. This means, that `num_stations` is equal to `len(segment_lengths)`.

To avoid inhomogeneous arrays, we will use a list of dictionaries and a parser function that turns it into proper flat arrays for Gurobi... This way, we should be able to extend the model and add new features using similar helper / parsing functionalities, as well as later in the constraint generation. 

In [ ]:
# Might use Enums or Dataclasses later but maybe also not xD
# class TrackType(Enum):
#     station = "station"
#     segment = "segment"

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [ ]:
# Parsing the track blueprint and generating necessary data

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [14]:
# Timetable data
time_horizon = 25

leave_times = [2, 3, 5]    # Due to the implementation ('looking-back') leave_times_i has to be >= 1
arrival_times = [18, 19, 20]

# Deriving some additional variables
num_trains = len(leave_times)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

In [ ]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x")